# Stage 1 — Earth Source Data QA

| Field | Value |
|---|---|
| **Pipeline stage** | Stage 1 — Earth source-data exploration |
| **Previous stage** | Stage 0 — Project setup (`channel_heads/io/paths.py`, `channel_heads/regimes.py`) |
| **Next stage** | Stage 2 — Earth interactive network exploration (`06_earth_network_explorer.ipynb`) |
| **Purpose** | Verify that all 17 Earth basin DEMs exist on disk, load correctly, have a valid CRS, and have sensible elevation ranges. |
| **Inputs** | `data/cropped_DEMs/*.tif` (RAW_KEEP) |
| **Outputs** | Console QA table; no files written. |
| **Decision gate** | All 17 DEMs must load. Any basin with a missing or corrupt DEM must be investigated before proceeding to Stage 5 feature generation. |

## 0. Configuration

In [ ]:
# Set to True to show a DEM thumbnail for every basin.
SHOW_THUMBNAILS = False

## 1. Imports and paths

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from channel_heads.io.paths import EXAMPLE_DEMS, CROPPED_DEMS_DIR
from channel_heads.basin_config import LOCAL_TO_PAPER_BASIN

print(f"DEM directory : {CROPPED_DEMS_DIR}")
print(f"Known basins  : {len(EXAMPLE_DEMS)}")

## 2. Per-basin file existence check

In [ ]:
rows = []
for basin, path in sorted(EXAMPLE_DEMS.items()):
    rows.append({
        "basin": basin,
        "paper_name": LOCAL_TO_PAPER_BASIN.get(basin, basin),
        "filename": path.name,
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1e6, 1) if path.exists() else None,
    })

exist_df = pd.DataFrame(rows)
missing = exist_df[~exist_df["exists"]]

print(f"Present : {exist_df['exists'].sum()}/{len(exist_df)}")
if not missing.empty:
    print(f"MISSING : {missing['basin'].tolist()}")

display(exist_df)

## 3. Load each DEM — check CRS, resolution, z-range

In [ ]:
try:
    import rasterio
    from rasterio.enums import Resampling
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    print("rasterio not available — install with: pip install rasterio")

qa_rows = []

if HAS_RASTERIO:
    for basin, path in sorted(EXAMPLE_DEMS.items()):
        row: dict = {"basin": basin, "ok": False, "error": None}
        if not path.exists():
            row["error"] = "FILE_MISSING"
            qa_rows.append(row)
            continue
        try:
            with rasterio.open(path) as src:
                crs = src.crs
                res_x, res_y = src.res
                w, h = src.width, src.height
                nodata = src.nodata
                data = src.read(1).astype(float)

            valid = data if nodata is None else data[data != nodata]
            row.update({
                "ok": True,
                "crs": str(crs),
                "is_projected": crs.is_projected if crs else None,
                "res_x_m": round(res_x, 1),
                "res_y_m": round(res_y, 1),
                "width_px": w,
                "height_px": h,
                "nodata": nodata,
                "z_min": round(float(np.nanmin(valid)), 1) if len(valid) else None,
                "z_max": round(float(np.nanmax(valid)), 1) if len(valid) else None,
                "z_range": round(float(np.nanmax(valid) - np.nanmin(valid)), 1) if len(valid) else None,
                "pct_nodata": round(100 * (1 - len(valid) / data.size), 2),
            })
        except Exception as exc:
            row["error"] = str(exc)
        qa_rows.append(row)

    qa_df = pd.DataFrame(qa_rows)

    failed = qa_df[~qa_df["ok"]]
    print(f"Loaded OK : {qa_df['ok'].sum()}/{len(qa_df)}")
    if not failed.empty:
        print(f"FAILED    : {failed[['basin','error']].to_string(index=False)}")

    display(qa_df.drop(columns=["error"], errors="ignore"))

## 4. CRS consistency check

In [ ]:
if HAS_RASTERIO and 'crs' in qa_df.columns:
    crs_counts = qa_df['crs'].value_counts()
    print("CRS distribution:")
    print(crs_counts.to_string())
    print()
    unprojected = qa_df[qa_df['is_projected'] == False]
    if not unprojected.empty:
        print(f"WARNING: {len(unprojected)} basin(s) have geographic (non-projected) CRS:")
        print(unprojected[['basin','crs']].to_string(index=False))
    else:
        print("All basins have projected CRS. ✓")

## 5. Elevation range summary

In [ ]:
if HAS_RASTERIO and 'z_range' in qa_df.columns:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 4))
    ok = qa_df[qa_df['ok']].sort_values('z_range', ascending=False)
    x = range(len(ok))
    ax.bar(x, ok['z_range'], label='z_range')
    ax.set_xticks(list(x))
    ax.set_xticklabels(ok['basin'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Elevation range (m)')
    ax.set_title('Earth basin elevation range (z_max − z_min)')
    fig.tight_layout()
    plt.show()

    print("\nSuspiciously flat basins (z_range < 50 m):")
    flat = ok[ok['z_range'] < 50]
    if flat.empty:
        print("  None.")
    else:
        print(flat[['basin','z_range']].to_string(index=False))

## 6. Optional DEM thumbnails

In [ ]:
if HAS_RASTERIO and SHOW_THUMBNAILS:
    import matplotlib.pyplot as plt

    basins_ok = sorted(EXAMPLE_DEMS.keys())
    ncols = 5
    nrows = -(-len(basins_ok) // ncols)  # ceiling division
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = axes.flatten()

    for i, basin in enumerate(basins_ok):
        ax = axes[i]
        path = EXAMPLE_DEMS[basin]
        if not path.exists():
            ax.set_title(f"{basin}\n(MISSING)", color='red')
            ax.axis('off')
            continue
        with rasterio.open(path) as src:
            scale = 256 / max(src.width, src.height)
            out_w = max(1, int(src.width * scale))
            out_h = max(1, int(src.height * scale))
            data = src.read(
                1, out_shape=(out_h, out_w),
                resampling=Resampling.bilinear
            ).astype(float)
            nd = src.nodata
        if nd is not None:
            data[data == nd] = np.nan
        lo, hi = np.nanpercentile(data, [2, 98])
        ax.imshow(data, cmap='terrain', vmin=lo, vmax=hi, aspect='auto')
        ax.set_title(basin, fontsize=8)
        ax.axis('off')

    for j in range(len(basins_ok), len(axes)):
        axes[j].axis('off')

    fig.suptitle('Earth DEM thumbnails (2–98th pct stretch)', y=1.01)
    fig.tight_layout()
    plt.show()
else:
    print("Set SHOW_THUMBNAILS = True to render DEM thumbnails.")

## 7. Gate summary

In [ ]:
if HAS_RASTERIO:
    n_ok = int(qa_df['ok'].sum())
    n_total = len(qa_df)
    if n_ok == n_total:
        print(f"Stage 1 QA PASSED — all {n_total} DEMs present, load, and have valid metadata.")
    else:
        failed_list = qa_df[~qa_df['ok']]['basin'].tolist()
        raise AssertionError(
            f"Stage 1 QA FAILED — {n_total - n_ok} DEM(s) did not load: {failed_list}\n"
            "Resolve missing or corrupt DEMs before proceeding to Stage 5."
        )
else:
    print("rasterio not available — install it to run the full QA gate.")